# Day 3: Machine Learning

## 1. Student Performance Classification (Decision Tree)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
df = pd.read_csv("student_performance.csv")
if "study_hours" not in df.columns:
    df["study_hours"] = df["studytime"] * 2.5
if "attendance" not in df.columns:
    df["attendance"] = 100 - (df["failures"] * 12)
if "assignments" not in df.columns:
    df["assignments"] = df["G1"]
if "result" not in df.columns:
    df["result"] = (df["G3"] >= 10).astype(int)
print(df.head())
print(df.groupby("result")[["study_hours", "attendance", "assignments"]].mean())
plt.figure(figsize=(7, 4))
plt.scatter(df["study_hours"], df["attendance"], c=df["result"])
plt.xlabel("Study hours")
plt.ylabel("Attendance")
plt.title("Student performance pattern")
plt.show()
X = df[["study_hours", "attendance", "assignments"]]
y = df["result"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Predictions:", predictions)
print("Accuracy:", accuracy_score(y_test, predictions))
plt.figure(figsize=(13, 7))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True
)
plt.title("Decision Tree Visualization")
plt.show()

## 2. MCU Movie Rating Classification (Decision Tree)

In [ ]:
df_mcu = pd.read_csv("mcu.csv")
print(df_mcu.head())
df_mcu["rating"] = (df_mcu["tomato_meter"] >= 75).astype(int)
print("Rating distribution:\n", df_mcu["rating"].value_counts())
print(
    df_mcu.groupby("rating")[[
        "audience_score",
        "movie_duration",
        "production_budget",
        "opening_weekend"
    ]].mean()
)
plt.figure(figsize=(7, 4))
plt.scatter(
    df_mcu["production_budget"],
    df_mcu["worldwide_box_office"],
    c=df_mcu["rating"]
)
plt.xlabel("Production Budget")
plt.ylabel("Worldwide Box Office")
plt.title("MCU Budget vs Box Office")
plt.grid(True)
plt.show()
X_mcu = df_mcu[[
    "audience_score",
    "movie_duration",
    "production_budget",
    "opening_weekend"
]]
y_mcu = df_mcu["rating"]
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mcu, y_mcu,
    test_size=0.20,
    random_state=42,
    stratify=y_mcu
)
model_mcu = DecisionTreeClassifier(max_depth=3, random_state=42)
model_mcu.fit(X_train_m, y_train_m)
y_pred_m = model_mcu.predict(X_test_m)
print("Accuracy:", accuracy_score(y_test_m, y_pred_m))
print("Confusion Matrix:\n", confusion_matrix(y_test_m, y_pred_m))
print("Classification Report:\n", classification_report(y_test_m, y_pred_m))
plt.figure(figsize=(16, 8))
plot_tree(
    model_mcu,
    feature_names=X_mcu.columns,
    class_names=["Low Rated", "High Rated"],
    filled=True
)
plt.title("MCU Rating Decision Tree")
plt.show()

## 3. Remote Sensing Image Analysis (ISRO LISS-III)

In [ ]:
from pathlib import Path
import matplotlib.image as mpimg
tif_files = sorted(Path(".").glob("BAND*.tif"))
if len(tif_files) >= 4:
    bands = [mpimg.imread(f).astype(np.float32) for f in tif_files[:4]]
else:
    np.random.seed(42)
    bands = [np.random.randint(20, 200, (200, 200)).astype(np.float32) for _ in range(4)]
G, R, NIR, SWIR = bands
fig, ax = plt.subplots(2, 2, figsize=(10, 8))
names = ["Band 1 (Green)", "Band 2 (Red)", "Band 3 (NIR)", "Band 4 (SWIR)"]
for a, b, name in zip(ax.ravel(), bands, names):
    a.imshow(b, cmap="gray")
    a.set_title(name)
    a.axis("off")
plt.tight_layout()
plt.show()
eps = 1e-6
ndvi = (NIR - R) / (NIR + R + eps)
rows, cols = np.indices(G.shape)
sample = pd.DataFrame({
    "row": rows.ravel(),
    "col": cols.ravel(),
    "green": G.ravel(),
    "red": R.ravel(),
    "nir": NIR.ravel(),
    "swir": SWIR.ravel(),
    "ndvi": ndvi.ravel()
})
print(sample.head())

## 4. Nearest Centroid Classifier from Scratch

In [ ]:
brightness = (G + R + NIR + SWIR) / 4.0
y_labels = np.full(len(sample), -1, dtype=int)
s_ndvi = sample["ndvi"].to_numpy()
s_red = sample["red"].to_numpy()
veg = (s_ndvi > 0.35)
water = (s_ndvi < 0.0) & (s_red < 50)
built = (s_ndvi >= 0.0) & (s_ndvi <= 0.25) & (s_red > 60)
y_labels[veg] = 0
y_labels[water] = 1
y_labels[built] = 2
sample["label"] = y_labels
labelled = sample[sample["label"] != -1].copy()
def fit_nearest_centroid(X, y):
    classes = np.unique(y)
    centroids = {}
    for c in classes:
        centroids[c] = X[y == c].mean(axis=0)
    return centroids
def predict_nearest_centroid(X, centroids):
    classes = sorted(centroids.keys())
    C = np.vstack([centroids[c] for c in classes])
    dists = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    idx = np.argmin(dists, axis=1)
    return np.array([classes[i] for i in idx])
feature_cols = ["green", "red", "nir", "swir", "ndvi"]
train_df = labelled.sample(frac=0.8, random_state=42)
test_df = labelled.drop(train_df.index)
centroids = fit_nearest_centroid(train_df[feature_cols].to_numpy(), train_df["label"].to_numpy())
pred = predict_nearest_centroid(test_df[feature_cols].to_numpy(), centroids)
classes = sorted(centroids.keys())
cm = np.zeros((len(classes), len(classes)), dtype=int)
for actual, predicted in zip(test_df["label"], pred):
    cm[int(actual), int(predicted)] += 1
print("Confusion Matrix:\n", pd.DataFrame(cm, index=classes, columns=classes))

## 5. Unsupervised Learning: K-Means Clustering from Scratch

In [ ]:
def kmeans_numpy(X, k=3, iterations=12, seed=7):
    rng = np.random.default_rng(seed)
    start = rng.choice(len(X), size=k, replace=False)
    centers = X[start].copy()
    for _ in range(iterations):
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = np.argmin(d2, axis=1)
        new_centers = np.array([X[labels == c].mean(axis=0) if np.any(labels == c) else centers[c] for c in range(k)])
        if np.allclose(centers, new_centers, atol=1e-4):
            break
        centers = new_centers
    return labels, centers
sub_X = sample[feature_cols].sample(min(2000, len(sample)), random_state=42).to_numpy()
cluster_labels, cluster_centers = kmeans_numpy(sub_X, k=3)
print("Cluster Centers:\n", cluster_centers)

## 6. Reinforcement Learning: Satellite Grid World

In [ ]:
step = max(1, int(max(NIR.shape) / 80))
small_ndvi = ((NIR - R) / (NIR + R + 1e-6))[::step, ::step]
small_ndvi = np.nan_to_num(small_ndvi, nan=0.0, posinf=0.0, neginf=0.0)
h, w = small_ndvi.shape
q_table = np.zeros((h, w, 4))
actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
print(f"Grid world size: {h}x{w}")
print("Q-table shape:", q_table.shape)